# **Lanche the server**

In [ ]:
!sudo apt install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess
import time

process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

time.sleep(5)
print("Ollama Server has bee launched")

Ollama Server has bee launched


In [ ]:
!ollama pull llama3

In [ ]:
!pip install langchain langgraph langchain-ollama
!pip install easyocr opencv-python-headless
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install numpy pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 23.5 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu118


# **First Version of Multi Agent Graph**

In [ ]:
import base64

image_path = "/content/drive/MyDrive/examen-national-maths-sciences-et-technologies-2023-normale-sujet_page-0002.jpg"

with open(image_path, "rb") as f:
    img_base64 = base64.b64encode(f.read()).decode()

In [ ]:
import easyocr
import cv2
import numpy as np
import time

# Decode Base64 to OpenCV image
def decode_image(img_base64):
    img_bytes = base64.b64decode(img_base64)
    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    return img

# Initialize OCR
reader = easyocr.Reader(['fr', 'en'], gpu=True)  # French + English

img = decode_image(img_base64)

start = time.time()
results = reader.readtext(img)
print("OCR finished in:", round(time.time() - start, 2), "seconds")

# Concatenate detected text
extracted_text = "\n".join([res[1] for res in results])
print("\n===== OCR Extracted Text =====\n")
print(extracted_text[:1000], "...")  # print first 1000 characters

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


OCR finished in: 30.6 seconds

===== OCR Extracted Text =====

Lagall
NS 22F
4s+snl _
2023 Zulul ; J9J) - Gslsa Jsall @ihsl jlaiyi
40
(LiàjLs) 44;
4Lua) UJIg p41
4lua -Clalji :5aL4
IIL
Exercice 1 (3points)
Dans lespace rapporté à un repère orthononmé direct
(0,,j,k),
on considère les points
4(0,1,4) , B(2,1,2)=
C(2,5,0) et 2(3,4,4) .
0.25
1) a) Montrer que AB
AC = 4(2i +j+2k
0.5
b) En déduire laire du triangle  ABC et la distance d (B ,( AC))
2) Soit D le milieu du segment [Ac]
0.25
a)
Vérifier que DS =
AB ^ Ac)
0.5
b) En déduire
d
'(2,(ABC)) =3 _
3) Soit (S) la sphère d'équation x2 +y? + 22 _ 6x
8y-82 +32 = 0
0.5
a) Déterminer le centre et le rayon de la sphère (S)
0.5
b) Montrer que le plan ( ABC) est tangent à la sphère (S) en un
que lon déterminera.
0.5
Soient (Q) et (9
les deux plans parallèles à ( ABC) tels que chacun deux coupe (S)
suivant un cercle de rayon v5
Détemminer une équation cartésienne pour chacun des deux plans (Q )
et
(9
Exercice 2 (Bpoints)
Dans le plan complexe ra

In [ ]:
# ================================
# FULL PIPELINE WITH DEBUG LOGS
# ================================

from typing import TypedDict, Optional
import easyocr
import cv2
import numpy as np
import base64
import json
import time

from langchain_ollama import ChatOllama

# ================================
# 1. STATE
# ================================

class AgentState(TypedDict):
    image_base64: str
    extracted_text: Optional[str]
    cleaned_text: Optional[str]
    solution: Optional[str]
    validation: Optional[str]
    is_valid: Optional[bool]
    retries: int

# ================================
# 2. CONFIG
# ================================

MAX_RETRIES = 2

llm = ChatOllama(
    model="llama3",  # change if needed
    temperature=0
)

reader = easyocr.Reader(['fr', 'en'], gpu=True)

# ================================
# DEBUG HELPER
# ================================

def debug_print(title):
    print("\n" + "="*20)
    print(f"🚀 {title}")
    print("="*20)

# ================================
# OCR AGENT
# ================================

def decode_image(img_base64):
    nparr = np.frombuffer(base64.b64decode(img_base64), np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    return img

def ocr_agent(state: AgentState):
    debug_print("OCR START")

    img = decode_image(state["image_base64"])

    start = time.time()
    results = reader.readtext(img)
    duration = time.time() - start

    text = "\n".join([res[1] for res in results])

    print(f"⏱ OCR time: {round(duration,2)} sec")
    print(f"📏 Extracted text length: {len(text)} chars")

    debug_print("OCR END")

    return {"extracted_text": text}

# ================================
# CLEANER AGENT
# ================================

def cleaner_agent(state: AgentState):
    debug_print("CLEANER START")

    print(f"📏 Input text length: {len(state['extracted_text'])}")

    prompt = f"""
Fix OCR errors, restore math, keep meaning. DO NOT solve.

TEXT:
{state['extracted_text']}
"""

    start = time.time()

    try:
        response = llm.invoke(prompt)
    except Exception as e:
        print("❌ Cleaner ERROR:", e)
        return {"cleaned_text": state["extracted_text"]}

    duration = time.time() - start

    cleaned = response.content

    print(f"⏱ Cleaner time: {round(duration,2)} sec")
    print(f"📏 Cleaned text length: {len(cleaned)}")

    debug_print("CLEANER END")

    return {"cleaned_text": cleaned}

# ================================
# SOLVER AGENT
# ================================

def solver_agent(state: AgentState):
    debug_print("SOLVER START")

    print(f"📏 Input text length: {len(state['cleaned_text'])}")

    prompt = f"""
Solve all problems step-by-step.

TEXT:
{state['cleaned_text']}
"""

    start = time.time()

    try:
        response = llm.invoke(prompt)
    except Exception as e:
        print("❌ Solver ERROR:", e)
        return {"solution": "ERROR"}

    duration = time.time() - start

    solution = response.content

    print(f"⏱ Solver time: {round(duration,2)} sec")
    print(f"📏 Solution length: {len(solution)}")

    debug_print("SOLVER END")

    return {"solution": solution}

# ================================
# VALIDATOR AGENT
# ================================

def validator_agent(state: AgentState):
    debug_print("VALIDATOR START")

    prompt = f"""
You are a strict validator.

Analyze the solution.

IMPORTANT:
- You MUST return ONLY valid JSON
- No text before or after
- No explanation outside JSON

FORMAT EXACTLY:
{{"is_valid": true, "feedback": "text"}}

TEXT:
{state['cleaned_text']}

SOLUTION:
{state['solution']}
"""

    start = time.time()

    try:
        response = llm.invoke(prompt)
        result = json.loads(response.content)
    except Exception as e:
        print("❌ Validator ERROR:", e)
        result = {"is_valid": False, "feedback": "Error parsing JSON"}

    duration = time.time() - start

    print(f"⏱ Validator time: {round(duration,2)} sec")
    print(f"📢 Feedback: {result['feedback']}")

    debug_print("VALIDATOR END")

    return {
        "validation": result["feedback"],
        "is_valid": result["is_valid"]
    }

# ================================
# MAIN PIPELINE
# ================================

def run_pipeline(image_path):
    debug_print("PIPELINE START")

    with open(image_path, "rb") as f:
        img_base64 = base64.b64encode(f.read()).decode()

    state: AgentState = {
        "image_base64": img_base64,
        "extracted_text": None,
        "cleaned_text": None,
        "solution": None,
        "validation": None,
        "is_valid": None,
        "retries": 0
    }

    # OCR
    state.update(ocr_agent(state))

    # CLEAN
    state.update(cleaner_agent(state))

    # LOOP
    while state["retries"] <= MAX_RETRIES:
        debug_print(f"ITERATION {state['retries']}")

        state.update(solver_agent(state))
        state.update(validator_agent(state))

        if state["is_valid"]:
            print("✅ Solution VALID")
            break

        state["retries"] += 1
        print(f"🔁 Retry {state['retries']}")

    debug_print("PIPELINE END")

    return state

# ================================
# RUN
# ================================

if __name__ == "__main__":
    result = run_pipeline("/content/drive/MyDrive/examen-national-maths-sciences-et-technologies-2023-normale-sujet_page-0002.jpg")

    print("\n===== FINAL CLEANED TEXT =====\n")
    print(result["cleaned_text"][:1000])

    print("\n===== FINAL SOLUTION =====\n")
    print(result["solution"][:1000])


🚀 PIPELINE START

🚀 OCR START
⏱ OCR time: 2.33 sec
📏 Extracted text length: 1611 chars

🚀 OCR END

🚀 CLEANER START
📏 Input text length: 1611
⏱ Cleaner time: 21.93 sec
📏 Cleaned text length: 1704

🚀 CLEANER END

🚀 ITERATION 0

🚀 SOLVER START
📏 Input text length: 1704
⏱ Solver time: 69.91 sec
📏 Solution length: 5275

🚀 SOLVER END

🚀 VALIDATOR START
⏱ Validator time: 5.22 sec
📢 Feedback: The solution is well-structured and easy to follow. The explanations are clear and concise, making it easy for the reader to understand the mathematical concepts. The formatting is good, with clear headings and proper use of whitespace. Overall, a great job!

🚀 VALIDATOR END
✅ Solution VALID

🚀 PIPELINE END

===== FINAL CLEANED TEXT =====

I'll do my best to fix the OCR errors and restore the math while keeping the meaning intact. Here's the corrected text:

TEXT:
Lagrange
NS 22F
Exercise 1 (3 points)
In the space referred to an orthonormal direct basis (0,7,i,k),
we consider the points A(0,1,4), B(2,1,2

In [ ]:
# ================================
# LANGGRAPH MULTI-AGENT SYSTEM (FIXED)
# ================================

from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama

import easyocr
import cv2
import numpy as np
import base64
import json
import time
import re

# ================================
# 1. STATE
# ================================

class AgentState(TypedDict):
    image_base64: str
    extracted_text: Optional[str]
    cleaned_text: Optional[str]
    solution: Optional[str]
    validation: Optional[str]
    is_valid: Optional[bool]
    retries: int

MAX_RETRIES = 2

# ================================
# 2. MODELS
# ================================

llm = ChatOllama(
    model="llama3",
    temperature=0
)

reader = easyocr.Reader(['fr', 'en'], gpu=True)

# ================================
# 3. HELPERS
# ================================

def debug(title):
    print("\n" + "="*20)
    print(f"🚀 {title}")
    print("="*20)

def decode_image(img_base64):
    nparr = np.frombuffer(base64.b64decode(img_base64), np.uint8)
    return cv2.imdecode(nparr, cv2.IMREAD_COLOR)

def extract_json(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                pass
    return {"is_valid": False, "feedback": "Invalid JSON"}

# ================================
# 4. AGENTS (NODES)
# ================================

def orchestrator_node(state: AgentState):
    debug("ORCHESTRATOR")
    return {}  # MUST return dict


def ocr_node(state: AgentState):
    debug("OCR")

    img = decode_image(state["image_base64"])
    results = reader.readtext(img)
    text = "\n".join([r[1] for r in results])

    print("📏 OCR length:", len(text))

    return {"extracted_text": text}


def cleaner_node(state: AgentState):
    debug("CLEANER")

    prompt = f"""
Fix OCR errors, restore math, keep meaning. DO NOT solve.

TEXT:
{state['extracted_text']}
"""

    start = time.time()
    response = llm.invoke(prompt)
    print("⏱ Cleaner:", round(time.time() - start, 2), "sec")

    return {"cleaned_text": response.content}


def solver_node(state: AgentState):
    debug("SOLVER")

    prompt = f"""
Solve all problems step-by-step but concisely.

TEXT:
{state['cleaned_text']}
"""

    start = time.time()
    response = llm.invoke(prompt)
    print("⏱ Solver:", round(time.time() - start, 2), "sec")

    return {"solution": response.content}


def validator_node(state: AgentState):
    debug("VALIDATOR")

    prompt = f"""
Check if solution is correct.

Return ONLY JSON:
{{"is_valid": true/false, "feedback": "..."}}

TEXT:
{state['cleaned_text']}

SOLUTION:
{state['solution']}
"""

    start = time.time()
    response = llm.invoke(prompt)
    result = extract_json(response.content)

    print("⏱ Validator:", round(time.time() - start, 2), "sec")
    print("📢 Feedback:", result["feedback"])

    return {
        "validation": result["feedback"],
        "is_valid": result["is_valid"]
    }

# ================================
# 5. ROUTER (DECISION LOGIC)
# ================================

def router(state: AgentState):
    if state.get("extracted_text") is None:
        return "ocr"

    if state.get("cleaned_text") is None:
        return "cleaner"

    if state.get("solution") is None:
        return "solver"

    if state.get("is_valid") is None:
        return "validator"

    # Retry logic
    if not state["is_valid"] and state["retries"] < MAX_RETRIES:
        print(f"🔁 Retry {state['retries'] + 1}")
        return "retry"

    return "end"

# ================================
# 6. RETRY NODE
# ================================

def retry_node(state: AgentState):
    return {"retries": state["retries"] + 1, "solution": None, "is_valid": None}

# ================================
# 7. BUILD GRAPH
# ================================

builder = StateGraph(AgentState)

# Nodes
builder.add_node("orchestrator", orchestrator_node)
builder.add_node("ocr", ocr_node)
builder.add_node("cleaner", cleaner_node)
builder.add_node("solver", solver_node)
builder.add_node("validator", validator_node)
builder.add_node("retry", retry_node)

# Entry
builder.set_entry_point("orchestrator")

# Routing
builder.add_conditional_edges(
    "orchestrator",
    router,
    {
        "ocr": "ocr",
        "cleaner": "cleaner",
        "solver": "solver",
        "validator": "validator",
        "retry": "retry",
        "end": END
    }
)

# Flow back to orchestrator
builder.add_edge("ocr", "orchestrator")
builder.add_edge("cleaner", "orchestrator")
builder.add_edge("solver", "orchestrator")
builder.add_edge("validator", "orchestrator")
builder.add_edge("retry", "orchestrator")

graph = builder.compile()

# ================================
# 8. RUN
# ================================

def run(image_path):
    with open(image_path, "rb") as f:
        img_base64 = base64.b64encode(f.read()).decode()

    state: AgentState = {
        "image_base64": img_base64,
        "extracted_text": None,
        "cleaned_text": None,
        "solution": None,
        "validation": None,
        "is_valid": None,
        "retries": 0
    }

    result = graph.invoke(state)

    print("\n===== FINAL CLEANED TEXT =====\n")
    print(result["cleaned_text"][:1000])

    print("\n===== FINAL SOLUTION =====\n")
    print(result["solution"][:1000])

    return result

# ================================
# EXECUTE
# ================================

if __name__ == "__main__":
    run("/content/drive/MyDrive/examen-national-maths-sciences-et-technologies-2023-normale-sujet_page-0002.jpg")


🚀 ORCHESTRATOR

🚀 OCR
📏 OCR length: 1611

🚀 ORCHESTRATOR

🚀 CLEANER
⏱ Cleaner: 21.29 sec

🚀 ORCHESTRATOR

🚀 SOLVER
⏱ Solver: 40.18 sec

🚀 ORCHESTRATOR

🚀 VALIDATOR
⏱ Validator: 3.26 sec
📢 Feedback: The solution appears to be correct and well-written. The steps are clear, and the math is accurate.

🚀 ORCHESTRATOR

===== FINAL CLEANED TEXT =====

I'll do my best to fix the OCR errors and restore the math while keeping the meaning intact. Here's the corrected text:

TEXT:
Lagrange
NS 22F
Exercise 1 (3 points)
In the space referred to an orthonormal direct basis (0,7,i,k),
we consider the points A(0,1,4), B(2,1,2) and C(2,5,0).
0.25
1) a) Show that AB = AC = 4(2i + j + 2k)
0.5
b) Deduce the area of triangle ABC and the distance between (B, AC)
2) Let D be the midpoint of segment [AC]
0.25
a) Verify that DS = AB × AC
0.5
b) Deduce that ∠(ABC) = 3 _

3) Let S be the sphere with equation x² + y² + z² - 6x - 8y + 82 + 32 = 0
0.5
a) Determine the center and radius of the sphere S
0.5
b) Show t